In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch9. RAG</font>

## RAG 구현 절차
1. docx 문서 로드 (python-docx : \n를 제외하고 읽어옴)
2. 문서 분할 (tiktoken 사용)
    - 모델의 context window와 token 비용 확인
    - chunk(분할된 문서 단위, 클수록 비용과 시간 소요 증가)로 분할
3. 분할된 문서 임베딩 → vector Store(DB)에 저장 ; Chroma(local), pinecone(cloud)
4. 입력 받은 질문과 vector Store에 저장된 문서의 유사도 검색
5. 유사도 상위 n개의 문서를 LLM에 질문과 함께 전달하여 답변 생성

# 1. 문서 로드 (python-docx)
- pip install -U -q langchain-community python-docx docx2txt

In [1]:
%pip install -U -q python-docx langchain-community docx2txt

Note: you may need to restart the kernel to use updated packages.


## 1.1 Docx2txtLoader

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax/소득세법(법률)(제21221호)(20260701).docx')
documents = loader.load() # \n을 포함하여 문서를 load
len(documents)

C:\Users\Admin\AppData\Local\Temp\ipykernel_24248\3129750862.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


1

## 1.2 Document

In [3]:
# 한 문단 단위로 load
from docx import Document
documents = Document('./tax/소득세법(법률)(제21221호)(20260701).docx')
print(documents)
print(dir(documents))

['_Document__body', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_block_width', '_body', '_element', '_parent', '_part', 'add_comment', 'add_heading', 'add_page_break', 'add_paragraph', 'add_picture', 'add_section', 'add_table', 'comments', 'core_properties', 'element', 'inline_shapes', 'iter_inner_content', 'paragraphs', 'part', 'save', 'sections', 'settings', 'styles', 'tables']


In [9]:
print(len(documents.paragraphs))
print(dir(documents.paragraphs))

3172
['__add__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__delitem__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__iadd__', '__imul__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__rmul__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']


In [7]:
for paragraphs in documents.paragraphs[20:30]:
    print(paragraphs.text)

5. “사업자”란 사업소득이 있는 거주자를 말한다.
② 제1항에 따른 주소ㆍ거소와 거주자ㆍ비거주자의 구분은 대통령령으로 정한다.
[본조신설 2009. 12. 31.]

제2조(납세의무) ① 다음 각 호의 어느 하나에 해당하는 개인은 이 법에 따라 각자의 소득에 대한 소득세를 납부할 의무를 진다.
1. 거주자
2. 비거주자로서 국내원천소득(國內源泉所得)이 있는 개인
② 다음 각 호의 어느 하나에 해당하는 자는 이 법에 따라 원천징수한 소득세를 납부할 의무를 진다.
1. 거주자
2. 비거주자


In [10]:
# \n이 제거된 문서 전체 내용 - full_text
full_text = ""
for paragraphs in documents.paragraphs:
    full_text += paragraphs.text + " "
len(full_text)

248301

# 2. 문서 분할 (tiktoken 사용)
- pip install tiktoken : OpenAI의 공식 tokenizer lib
    - full_text → token 단위 문자 → 정수 → 문자
    - 1500 token 단위로 문서 분할

In [12]:
%pip install -q tiktoken

Note: you may need to restart the kernel to use updated packages.


In [15]:
import tiktoken
encoder = tiktoken.encoding_for_model('gpt-4o-mini')

# full_text → 정수 list
encoding = encoder.encode(full_text)

# 정수 list → 문자
decoding = encoder.decode(encoding)

In [18]:
print(f'글자 수 : {len(full_text)}')
print(f'토큰 수 : {len(encoding)}')
print(f'문자 : {decoding[:10]}')
print(f'숫자 : {encoding[:10]}')

글자 수 : 248301
토큰 수 : 162419
문자 :  소득세법 [시행 
숫자 : [22183, 64328, 11734, 22070, 723, 5637, 15719, 220, 1323, 21]


In [19]:
print(list(range(0, 162419, 1500)))

[0, 1500, 3000, 4500, 6000, 7500, 9000, 10500, 12000, 13500, 15000, 16500, 18000, 19500, 21000, 22500, 24000, 25500, 27000, 28500, 30000, 31500, 33000, 34500, 36000, 37500, 39000, 40500, 42000, 43500, 45000, 46500, 48000, 49500, 51000, 52500, 54000, 55500, 57000, 58500, 60000, 61500, 63000, 64500, 66000, 67500, 69000, 70500, 72000, 73500, 75000, 76500, 78000, 79500, 81000, 82500, 84000, 85500, 87000, 88500, 90000, 91500, 93000, 94500, 96000, 97500, 99000, 100500, 102000, 103500, 105000, 106500, 108000, 109500, 111000, 112500, 114000, 115500, 117000, 118500, 120000, 121500, 123000, 124500, 126000, 127500, 129000, 130500, 132000, 133500, 135000, 136500, 138000, 139500, 141000, 142500, 144000, 145500, 147000, 148500, 150000, 151500, 153000, 154500, 156000, 157500, 159000, 160500, 162000]


In [20]:
import tiktoken

def split_text(full_text, chunk_size=1500):
    'full_text를 chunk_size로 분할하여 chunk_list로 반환'
    encoder = tiktoken.encoding_for_model('gpt-4o-mini')
    total_encoding = encoder.encode(full_text)
    total_token_count = len(total_encoding)
    
    chunk_list = []
    for i in range(0, total_token_count, chunk_size):
        chunk_encoding = total_encoding[i : i+chunk_size]
        chunk_decoded = encoder.decode(chunk_encoding)
        chunk_list.append(chunk_decoded)
    return chunk_list

In [21]:
chunk_list = split_text(full_text, 1500)
len(chunk_list)

109

# 3. 분할된 문서 임베딩
- pip install chromadb
- vector Store(DB)에 저장 ; Chroma(local), pinecone(cloud)

In [27]:
%pip install -q chromadb --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


In [28]:
import chromadb
chroma_client = chromadb.Client()

In [29]:
# collection : RDB의 TABLE 역할
collection_name = 'tax_collection'

In [30]:
# 임베딩 객체 생성
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
load_dotenv()
import os
openai_key = os.getenv('OPEN_API_KEY')
openai_embedding = OpenAIEmbeddingFunction(
    api_key = openai_key,
    model_name='text-embedding-3-large'
)

In [31]:
# collection 입력 시 사용할 id
ids = [str(id) for id in range(len(chunk_list))]

In [32]:
# chromaDB(inmemory;변수)에 collection 생성
tax_collection = chroma_client.get_or_create_collection(
    name = collection_name,
    embedding_function=openai_embedding
)

In [33]:
# chromaDB의 collection에 chunk_list 저장 (인메모리)
tax_collection.add(documents=chunk_list, ids=ids)

In [39]:
# tax_collection 확인
results = tax_collection.get(include=['documents', 'embeddings', 'metadatas'])

# 4. 유사도 검색

In [40]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_doc = tax_collection.query(
    query_texts=query,
    n_results=3 # 유사도 상위 n개 반환 ; 기본값 : 10
)

In [41]:
retrieved_doc['distances']

[[0.553639829158783, 0.5724149346351624, 0.5735224485397339]]

# 5. LLM을 통한 답변 생성

In [43]:
from openai import OpenAI
load_dotenv()
client = OpenAI()
response = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages=[
        {'role':'system', 
         'content':f"""You are an expert on South Korean income tax.Please answer the user's question based on the information below.
         {retrieved_doc['documents'][0]}"""},
        {'role':'user','content':query}
    ]
)

In [45]:
print(response.choices[0].message.content)

대한민국의 소득세는 여러 구간의 세율에 따라 누진적으로 과세됩니다. 2023년 기준으로 소득세 세율은 다음과 같습니다:

1. 1,200만원 이하: 6%
2. 1,200만원 초과 ~ 4,600만원 이하: 15%
3. 4,600만원 초과 ~ 8,800만원 이하: 24%

직장인의 연봉이 5,000만원일 경우 계산해보면 다음과 같습니다.

1. 첫 번째 구간 (1,200만원 이하):
   - 1,200만원 × 6% = 72만원

2. 두 번째 구간 (1,200만원 초과 ~ 4,600만원 이하):
   - (4,600만원 - 1,200만원) × 15% = 5,400만원 × 15% = 810만원

3. 세 번째 구간 (4,600만원 초과 ~ 5,000만원 이하):
   - (5,000만원 - 4,600만원) × 24% = 400만원 × 24% = 96만원

이제 모든 구간의 세금을 합산하면:

- 72만원 + 810만원 + 96만원 = 978만원

따라서 연봉 5,000만원인 직장인의 소득세는 약 978만원입니다. 

추가적으로 세액공제를 고려하면 최종 세액은 달라질 수 있습니다.
